# 04 — Bring Your Own SAM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/04_bring_your_own_sam.ipynb)

This notebook shows the data pipeline:

```text
one balanced SAM.csv
      ↓
identify factors + institutional accounts
      ↓
CGE-Core derives the goods set
      ↓
build_dataset(...)
      ↓
solve the standard-CGE benchmark
      ↓
branch a policy Scenario
```

The default demonstration starts from the bundled standard SAM, relabels its institutional accounts, and rebuilds the model dataset from a single CSV.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# In Colab, set the environment variable CGE_CORE_REF to test a branch or tag.
# The public notebooks default to main. Outside Colab, a current CGE-Core git
# checkout is used directly, so branch development never silently resets to main.
CGE_CORE_REF = os.environ.get("CGE_CORE_REF", "main")
IN_COLAB = Path("/content").exists()

if not IN_COLAB and (Path.cwd() / ".git").is_dir() and (Path.cwd() / "cge_core").is_dir():
    REPO_DIR = Path.cwd()
    source_label = "current checkout"
else:
    WORKSPACE = Path("/content") if IN_COLAB else Path.home() / ".cache"
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    REPO_DIR = WORKSPACE / "CGE-core-colab"
    REPO_URL = "https://github.com/miraflor/CGE-core.git"

    if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a git checkout.")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--no-checkout", REPO_URL, str(REPO_DIR)],
            check=True,
        )

    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", CGE_CORE_REF, "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"],
        check=True,
    )
    source_label = CGE_CORE_REF

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cge_core

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
    text=True,
).strip()
print("✓ CGE-Core", cge_core.__version__)
print("✓ Source:", source_label, f"({commit})")
print("✓ Repository:", REPO_DIR)


import shutil

if not shutil.which("ipopt"):
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "amplpy"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "amplpy.modules", "install", "coin"],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    module_path = subprocess.check_output(
        [sys.executable, "-m", "amplpy.modules", "path"],
        text=True,
    ).strip()
    os.environ["PATH"] = module_path + os.pathsep + os.environ.get("PATH", "")

assert shutil.which("ipopt"), "IPOPT was not found."
SOLVER = "ipopt"
print("✓ Solver:", SOLVER)


## 2. Create a single-SAM input file

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

from cge_core import CGE, example_data, samtools
from cge_core.models import StdCGE

source_sam = example_data("stdcge") / "param-sam-.csv"
sam = pd.read_csv(source_sam, index_col=0)

rename = {
    "HOH": "HH",
    "GOV": "GOVT",
    "INV": "SAV-INV",
    "EXT": "ROW",
    "IDT": "ITAX",
    "TRF": "TARIFF",
}
sam = sam.rename(index=rename, columns=rename)

MY_SAM = Path("/content/my_sam.csv") if Path("/content").exists() else Path("my_sam.csv")
sam.to_csv(MY_SAM, index_label="U")

display(sam)
print("Saved single-SAM input:", MY_SAM)

## 3. Tell CGE-Core which accounts play which roles

For the standard model, CGE-Core needs the factor accounts plus the household, government, saving/investment, rest-of-world, indirect-tax, and tariff accounts. Every remaining account becomes a good/sector.

In [ ]:
accounts = dict(
    hoh="HH",
    gov="GOVT",
    inv="SAV-INV",
    ext="ROW",
    idt="ITAX",
    trf="TARIFF",
)

OUT_DIR = Path("/content/my_cge_data") if Path("/content").exists() else Path("my_cge_data")

samtools.build_dataset(
    MY_SAM,
    OUT_DIR,
    factors=["CAP", "LAB"],
    institutions=accounts.values(),
)

print("Generated:")
for path in sorted(OUT_DIR.iterdir()):
    print(" ", path.name)

## 4. Load and solve the relabelled benchmark economy

In [ ]:
factors = ["CAP", "LAB"]
institutions = set(accounts.values())
goods = [account for account in sam.index if account not in set(factors) | institutions]

model = CGE(model=StdCGE(accounts=accounts), data=OUT_DIR)
benchmark = model.solve_benchmark(
    numeraire=("pf", "LAB"),
    redundant=("eqpf", "LAB"),
    solver=SOLVER,
)

print("Derived goods:", goods)
print("Factors:", factors)
print("Institution labels:", accounts)
print("Benchmark welfare objective:", benchmark.objective)

## 5. Prove that it is a live model

In [ ]:
scenario = benchmark.scenario("Abolish MLK tariff")
scenario.set("taum", "MLK", 0.0)
result = scenario.solve(solver=SOLVER)
results = result.compare(benchmark)

display(
    results[
        results["component"].isin(["Z", "M", "E", "Xp", "pq"])
    ][["component", "index_1", "reference_value", "value", "pct_change"]]
    .style.format({
        "reference_value": "{:.4f}",
        "value": "{:.4f}",
        "pct_change": "{:+.2f}%",
    })
)

## 6. Use an actual SAM in Colab

Set `USE_UPLOAD = True`, rerun the cell, and select a CSV from your computer.

The file must be square, finite, balanced, and have the account structure required by the standard model. A balanced arbitrary SAM is **not automatically behaviorally compatible** with `stdcge`.

In [ ]:
USE_UPLOAD = False

if USE_UPLOAD:
    from google.colab import files

    uploaded = files.upload()
    uploaded_name = next(iter(uploaded))
    uploaded_path = Path("/content") / uploaded_name
    print("Uploaded:", uploaded_path)

    # Then adapt the labels below to your SAM:
    #
    # samtools.build_dataset(
    #     uploaded_path,
    #     "/content/uploaded_cge_data",
    #     factors=["CAP", "LAB"],
    #     institutions=accounts.values(),
    # )
else:
    print("Upload step skipped. Set USE_UPLOAD = True when you have your own SAM.")

## What you learned

**Calibration is where data and model meet.**

The SAM supplies the benchmark accounting flows. The model supplies the behavioral structure. Calibration recovers parameters that make the behavioral model reproduce the benchmark economy.

## Next

Notebook 05 moves to CGE-Core's richer **IFPRI Standard CGE subsystem**.

[Open Notebook 05 in Colab](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/05_ifpri_standard_cge.ipynb)